In [5]:
# rag_local.py
# CHANGED (Cell 0): the notebook's `!pip ...` shell line is not valid Python.
# Install once from your VS Code terminal instead (kept identical package set):
#   pip install -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu pypdf huggingface_hub

# =============== (was Cell 1) local corpus path ===============
# CHANGED: removed `from google.colab import drive` + `drive.mount(...)` (Colab-only).
# CHANGED: PDF_DIR now points to a local folder. Set this to wherever your PDFs are.
PDF_DIR = r"/Users/alejandrogomez-paz/Desktop/RAG/mini-corpus/docs"   # CHANGED: local path

# =============== (Cell 2) ingest, chunk, embed, FAISS — UNCHANGED ===============
import os, glob, re
from pypdf import PdfReader
from LLM_local import tokenizer, model
# CHANGED: must be set BEFORE importing torch/faiss/transformers
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"       # stop faiss+torch double-loading OpenMP (macOS crash)
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"    # unsupported MPS ops fall back to CPU instead of crashing
os.environ["TOKENIZERS_PARALLELISM"] = "false"     # silence/avoid tokenizer fork issues

def clean_text(t: str) -> str:
    t = t.replace("\x00", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def chunk_text(text: str, chunk_chars=2200, overlap=300):
    # simple, reliable char chunking
    chunks = []
    i = 0
    n = len(text)
    while i < n:
        chunk = text[i:i+chunk_chars].strip()
        if chunk:
            chunks.append(chunk)
        i += (chunk_chars - overlap)
    return chunks

pdf_paths = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
print("PDFs found:", len(pdf_paths))

chunks = []
metas  = []

for path in pdf_paths:
    try:
        reader = PdfReader(path)
        for pi, page in enumerate(reader.pages):
            txt = page.extract_text() or ""
            txt = clean_text(txt)
            if not txt:
                continue
            for c in chunk_text(txt):
                chunks.append(c)
                metas.append({
                    "source": os.path.basename(path),
                    "page": pi + 1
                })
    except Exception as e:
        print("Failed:", path, e)

print("Total chunks:", len(chunks))
print("Example meta:", metas[0] if metas else None)

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

emb = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
emb = np.asarray(emb, dtype=np.float32)

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)   # cosine similarity if embeddings normalized
index.add(emb)

print("FAISS index size:", index.ntotal)

# =============== (was Cell 3) HF login ===============
# CHANGED: made login optional. Only needed for gated models (e.g. Mistral/Gemma).
# Uncomment and run once, OR set HUGGINGFACE_HUB_TOKEN in your env, OR use an open model below.
# from huggingface_hub import login
# login()


# =============== (Cell 4) load LLM — STABLE CPU PATH ===============
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =============== (Cell 5) retrieve + rag_answer — UNCHANGED ===============
def retrieve(question: str, k: int = 5):
    qv = embedder.encode([question], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(qv, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        results.append((float(score), chunks[idx], metas[idx]))
    return results

def rag_answer(question: str, k: int = 5, max_new_tokens: int = 300):
    results = retrieve(question, k=k)

    context_blocks = []
    for score, txt, meta in results:
        context_blocks.append(
            f"[SOURCE: {meta['source']} | page {meta['page']} | score {score:.3f}]\n{txt}"
        )
    context = "\n\n---\n\n".join(context_blocks)

    system = (
        "You are RAD-AI, a radiation safety assistant.\n"
        "Use ONLY the provided context to answer.\n"
        "If the answer is not in the context, say: "
        "'I don't have enough information in the provided files.'\n"
        "Cite sources as (source, page)."
    )

    user = f"CONTEXT:\n{context}\n\nQUESTION:\n{question}"

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    else:
        prompt = system + "\n\n" + user + "\n\nANSWER:"
        inputs = tokenizer(prompt, return_tensors="pt")

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)

# CHANGED: wrapped the demo call in __main__ so importing this file doesn't auto-run generation.


ImportError: cannot import name 'tokenizer' from 'LLM_local' (/Users/alejandrogomez-paz/Desktop/RAG/LLM_local.py)

In [2]:
from huggingface_hub import login
login()   # Paste your HF token after accepting Gemma's license

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print(model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 288/288 [00:10<00:00, 27.98it/s]


mps:0


if __name__ == "__main__":
    print(rag_answer(
        "Give me a 3-bullet summary of the nuclear safety regulatory framework mentioned in these files.",
        k=8
    ))

In [ ]:
print(rag_answer("What is the dose threshold for acute radiation syndrome?", k=5))

NameError: name 'rag_answer' is not defined